# 1. Procesamiento y Análisis de Datos
## Laboratorio de Ciencia de Datos ADA — EPN
Dataset: Alquileres de bienes inmuebles en Ecuador

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re, warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')

## 1. Carga del Dataset

In [ ]:
df = pd.read_csv('../data/real_state_ecuador_dataset.csv')
print(f"Forma: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Limpieza y Normalización de la columna `Lugar`

El campo `Lugar` contiene cadenas con el formato `'Provincia, [barrio/dirección], Ciudad, Ecuador'`. Extraemos la ciudad como el penúltimo token.

In [ ]:
def extraer_ciudad(lugar):
    if pd.isna(lugar):
        return np.nan
    partes = [p.strip() for p in lugar.split(',')]
    partes = [p for p in partes if p.lower() not in ('ecuador',)]
    return partes[-1].strip() if partes else np.nan

def normalizar_ciudad(ciudad):
    if pd.isna(ciudad):
        return np.nan
    if re.match(r'^Quito\s+\d+', ciudad):
        return 'Quito'
    return {'Sangolqui': 'Sangolquí'}.get(ciudad, ciudad)

df['Lugar_original'] = df['Lugar']
df['Lugar'] = df['Lugar'].apply(extraer_ciudad).apply(normalizar_ciudad)
print(f"Valores únicos en Lugar: {df['Lugar'].nunique()}")
df['Lugar'].value_counts().head(15)

## 3. Manejo de Valores Faltantes

In [ ]:
print("Nulos antes:")
print(df.isnull().sum())

# Limpiar Num. dormitorios
df['Num. dormitorios'] = df['Num. dormitorios'].replace('1 HABITACIÓN', '1')
df['Num. dormitorios'] = pd.to_numeric(df['Num. dormitorios'], errors='coerce')

# Imputar con mediana por Lugar, fallback a mediana global
for col in ['Num. dormitorios', 'Num. banos', 'Area', 'Num. garages']:
    med_lugar  = df.groupby('Lugar')[col].transform('median')
    med_global = df[col].median()
    df[col] = df[col].fillna(med_lugar).fillna(med_global)

print("\nNulos después:")
print(df.isnull().sum())

In [ ]:
# Eliminar outliers extremos de precio (top 1%)
p99 = df['Precio'].quantile(0.99)
df_clean = df[df['Precio'] <= p99].copy()
print(f"Registros tras filtrar outliers: {len(df_clean)} (de {len(df)})")

## 4. Análisis Descriptivo
### 4.1 Total de propiedades por Provincia y por Lugar

In [ ]:
print("=== Por Provincia ===")
print(df_clean['Provincia'].value_counts().to_string())
print("\n=== Top 20 Lugares ===")
print(df_clean['Lugar'].value_counts().head(20).to_string())

### 4.2 Mediana y Promedio de Precio (General y por Lugar)

In [ ]:
print(f"Promedio global : ${df_clean['Precio'].mean():.2f}")
print(f"Mediana global  : ${df_clean['Precio'].median():.2f}\n")

stats = (df_clean.groupby('Lugar')['Precio']
         .agg(Promedio='mean', Mediana='median', N='count')
         .sort_values('Mediana', ascending=False))
stats.round(2).head(15)

### 4.3 Relación Área – Precio

In [ ]:
corr = df_clean[['Area','Precio']].corr().iloc[0,1]
print(f"Correlación de Pearson Área-Precio: {corr:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df_clean['Area'], df_clean['Precio'], alpha=0.4, s=15, color='steelblue')
z = np.polyfit(df_clean['Area'].dropna(), df_clean.loc[df_clean['Area'].notna(),'Precio'], 1)
xs = np.linspace(df_clean['Area'].min(), df_clean['Area'].max(), 100)
ax.plot(xs, np.poly1d(z)(xs), 'r--', label=f'r = {corr:.2f}')
ax.set_xlabel('Área (m²)'); ax.set_ylabel('Precio (USD)')
ax.set_title('Área vs Precio'); ax.legend()
plt.tight_layout(); plt.show()

### 4.4 Premium por Habitación Adicional

In [ ]:
print("Premium por habitación adicional:")
for h in range(1, 6):
    g1 = df_clean[df_clean['Num. dormitorios']==h]['Precio'].mean()
    g2 = df_clean[df_clean['Num. dormitorios']==h+1]['Precio'].mean()
    if not (np.isnan(g1) or np.isnan(g2)):
        pct = (g2/g1 - 1)*100
        print(f"  {h} → {h+1} hab: +${g2-g1:.2f} ({pct:+.1f}%)")

### 4.5 Distribución de precios y otras visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Análisis Exploratorio – Inmuebles en Alquiler Ecuador', fontsize=13, fontweight='bold')

# Distribución precio
ax = axes[0,0]
ax.hist(df_clean['Precio'], bins=40, color='steelblue', edgecolor='white')
ax.axvline(df_clean['Precio'].median(), color='red',    ls='--', label=f"Mediana: ${df_clean['Precio'].median():.0f}")
ax.axvline(df_clean['Precio'].mean(),   color='orange', ls='--', label=f"Promedio: ${df_clean['Precio'].mean():.0f}")
ax.set_title('Distribución de Precio'); ax.set_xlabel('USD'); ax.legend()

# Por provincia
ax = axes[0,1]
pp = df_clean['Provincia'].value_counts().head(8)
ax.barh(pp.index[::-1], pp.values[::-1], color='teal')
ax.set_title('Propiedades por Provincia (Top 8)')

# Precio por dormitorios
ax = axes[1,0]
hab = df_clean[df_clean['Num. dormitorios'].between(1,6)].groupby('Num. dormitorios')['Precio'].mean()
ax.bar(hab.index.astype(int), hab.values, color='steelblue')
ax.set_title('Precio Promedio por Dormitorios'); ax.set_xlabel('Dormitorios')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Precio por baños
ax = axes[1,1]
ban = df_clean[df_clean['Num. banos'].between(1,5)].groupby('Num. banos')['Precio'].mean()
ax.bar(ban.index.astype(float).astype(str), ban.values, color='steelblue')
ax.set_title('Precio Promedio por Baños'); ax.set_xlabel('Baños')

plt.tight_layout(); plt.show()

## 5. Nueva Columna: Tipo de Precio por Lugar

In [ ]:
def clasificar_precio(grupo):
    q1 = grupo['Precio'].quantile(0.25)
    q3 = grupo['Precio'].quantile(0.75)
    return grupo['Precio'].apply(lambda p: 'Económico' if p < q1 else ('Lujo' if p > q3 else 'Medio'))

df_clean['Tipo_Precio'] = df_clean.groupby('Lugar', group_keys=False).apply(clasificar_precio)
print(df_clean['Tipo_Precio'].value_counts())
df_clean[['Precio','Lugar','Tipo_Precio']].sample(10)

In [ ]:
# Guardar dataset limpio
df_clean.to_csv('../data/real_state_clean.csv', index=False)
print("Dataset limpio guardado en data/real_state_clean.csv")